# Little Lemon — Exploratory Data Analysis
This notebook profiles the supplied orders workbook, prepares a Tableau-ready extract, and generates summary views. It intentionally flags date anomalies instead of silently correcting them.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

source_file = '../data/LittleLemon_Data.xlsx'
df = pd.read_excel(source_file)
df.head()

## 1. Inspect structure and data quality

In [ ]:
print('Shape:', df.shape)
df.info()
display(df.isna().sum().to_frame('missing_values'))
display(df.describe(include='all').T)

## 2. Standardize fields and derive measures

In [ ]:
data = df.copy()
data.columns = [c.strip().lower().replace(' ', '_').replace('-', '_') for c in data.columns]
data = data.rename(columns={'desert_name':'dessert_name'})
data['country_code'] = data['country_code'].fillna('UNK')
data['gross_profit'] = data['sales'] - data['cost'] - data['delivery_cost']
data['order_year'] = data['order_date'].dt.year
data['order_month'] = data['order_date'].dt.to_period('M').astype(str)
data.head()

## 3. Profile orders, customers, geography and dates

In [ ]:
print('Rows:', len(data))
print('Distinct order IDs:', data['order_id'].nunique())
print('Distinct customers:', data['customer_id'].nunique())
print('Countries:', data['country'].nunique())
print('Delivery date earlier than order date:', (data['delivery_date'] < data['order_date']).sum())

## 4. Sales by month and cuisine

In [ ]:
monthly = data.groupby('order_month', as_index=False)['sales'].sum()
monthly.plot(x='order_month', y='sales', figsize=(12,4), marker='o', title='Sales by Order Month', legend=False)
plt.xticks(rotation=60); plt.ylabel('Sales (source currency units)'); plt.tight_layout(); plt.show()

data.groupby('cuisine_name')['sales'].sum().sort_values(ascending=False).plot(kind='bar', title='Sales by Cuisine', figsize=(7,4))
plt.ylabel('Sales'); plt.tight_layout(); plt.show()

## 5. Top countries and course quantity

In [ ]:
data.groupby('country')['sales'].sum().nlargest(10).sort_values().plot(kind='barh', title='Top 10 Countries by Sales', figsize=(8,5))
plt.xlabel('Sales'); plt.tight_layout(); plt.show()

data.groupby('course_name')['quantity'].sum().sort_values(ascending=False).plot(kind='bar', title='Quantity by Course', figsize=(7,4))
plt.ylabel('Quantity'); plt.tight_layout(); plt.show()

## 6. Export cleaned data for Tableau
The source workbook does not specify currency. Keep the financial measures in source units. Country-code nulls are replaced with UNK. Date anomalies remain unchanged and should be investigated with the data owner.

In [ ]:
data.to_csv('../data/little_lemon_orders_clean.csv', index=False, encoding='utf-8-sig')
print('Exported', len(data), 'rows.')